# 3) The DC approximation — Exercises

**Goals:** write a DC power flow from a line table, estimate losses from DC flows, use DC angles to start an AC solver, find an overload that DC cannot see, and test whether DC ranks line outages correctly.

The setup cell holds the lecture's Croatian network (`croatia_pandapower` with switches for the Zagreb voltage set point, line ratings and an outage; `croatia_pypsa`), and lesson 1's `case14_data` and `newton_raphson`. Power in MW, 100 MVA base.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import pypsa

warnings.filterwarnings("ignore")
for name in ("pandapower", "pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

# ---- the lecture's stylised Croatian 400 kV network, with a few switches for the exercises ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
R_KM, X_KM, C_KM = 0.028, 0.32, 11.5
LOAD = {"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250}
GEN = {"Rijeka": 500, "Split": 600, "Velebit": 300}


def croatia_pandapower(v_zagreb=1.0, rating_ka=None, out_of_service=None):
    """The lecture's network. rating_ka: {line position: kA}; out_of_service: line position to switch off."""
    net = pp.create_empty_network(sn_mva=100)
    bus = {name: pp.create_bus(net, 400, name=name) for name in BUSES}
    pp.create_ext_grid(net, bus["Zagreb"], vm_pu=v_zagreb)
    for name, p in GEN.items():
        pp.create_gen(net, bus[name], p_mw=p, vm_pu=1.0)
    for name, p in LOAD.items():
        pp.create_load(net, bus[name], p_mw=p, q_mvar=0.2 * p)
    for i, line in LINES.iterrows():
        pp.create_line_from_parameters(net, bus[line.bus0], bus[line.bus1], line.km, R_KM, X_KM, C_KM,
                                       max_i_ka=(rating_ka or {}).get(i, 2.0), in_service=(i != out_of_service))
    return net


def croatia_pypsa():
    n = pypsa.Network()
    n.add("Bus", BUSES, v_nom=400)
    n.add("Generator", "slack", bus="Zagreb", control="Slack")
    for name, p in GEN.items():
        n.add("Generator", f"{name} gen", bus=name, p_set=p, control="PV")
    for name, p in LOAD.items():
        n.add("Load", f"{name} load", bus=name, p_set=p, q_set=0.2 * p)
    for _, line in LINES.iterrows():
        n.add("Line", f"{line.bus0}-{line.bus1}", bus0=line.bus0, bus1=line.bus1, r=R_KM * line.km, x=X_KM * line.km,
              b=2 * np.pi * 50 * C_KM * 1e-9 * line.km, s_nom=np.sqrt(3) * 400 * 2.0)
    return n


# ---- lesson 1's Newton-Raphson and the IEEE 14-bus internal data ----
def case14_data(load_scale=1.0):
    net = pn.case14()
    net.load[["p_mw", "q_mvar"]] *= load_scale
    net.gen.p_mw *= load_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10)
    internal, order = net._ppc["internal"], net._pd2ppc_lookups["bus"]
    v_set = np.ones(len(internal["Sbus"]))
    for table in (net.gen, net.ext_grid):
        v_set[order[table.bus.to_numpy()]] = table.vm_pu.to_numpy()
    return net, {"Y": internal["Ybus"].toarray(), "S": internal["Sbus"].copy(), "v_set": v_set, "pv": internal["pv"],
                 "pq": internal["pq"], "order": order}


def newton_raphson(Y, S, V0, pv, pq, tol=1e-10, max_iter=30):
    """Lesson 1's solver. Returns V and the number of iterations."""
    V = V0.astype(complex).copy()
    pvpq = np.r_[pv, pq]
    for it in range(max_iter + 1):
        I = Y @ V
        m = V * np.conj(I) - S
        F = np.r_[m[pvpq].real, m[pq].imag]
        if np.abs(F).max() < tol:
            return V, it
        Vn = V / np.abs(V)
        dS_dVa = 1j * np.diag(V) @ np.conj(np.diag(I) - Y @ np.diag(V))
        dS_dVm = np.diag(V) @ np.conj(Y @ np.diag(Vn)) + np.conj(np.diag(I)) @ np.diag(Vn)
        J = np.block([[dS_dVa[np.ix_(pvpq, pvpq)].real, dS_dVm[np.ix_(pvpq, pq)].real],
                      [dS_dVa[np.ix_(pq, pvpq)].imag, dS_dVm[np.ix_(pq, pq)].imag]])
        dx = np.linalg.solve(J, -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError("no convergence")

### Warm-ups

**1. A DC power flow from the line table**

Write `dc_flows()` for the Croatian network directly from `LINES`, `LOAD` and `GEN`: convert each line's reactance to per unit (x_pu = x_Ω / (400² / 100)), build B, fix the slack angle at Zagreb to zero, solve for the other angles and return the line flows in MW. Check them against PyPSA's `lpf()`.

In [2]:
def dc_flows() -> pd.Series:
    """DC line flows (MW) of the Croatian network, from bus0 to bus1, indexed like PyPSA's lines."""
    ### BEGIN SOLUTION
    idx = {name: k for k, name in enumerate(BUSES)}
    x_pu = X_KM * LINES.km.to_numpy() / (400 ** 2 / 100)
    b = 1 / x_pu
    f = LINES.bus0.map(idx).to_numpy()
    t = LINES.bus1.map(idx).to_numpy()
    n = len(BUSES)
    B = np.zeros((n, n))
    np.add.at(B, (f, f), b); np.add.at(B, (t, t), b)
    np.add.at(B, (f, t), -b); np.add.at(B, (t, f), -b)
    P = np.zeros(n)
    for name, p in GEN.items():
        P[idx[name]] += p
    for name, p in LOAD.items():
        P[idx[name]] -= p
    keep = [k for k in range(n) if k != idx["Zagreb"]]
    theta = np.zeros(n)
    theta[keep] = np.linalg.solve(B[np.ix_(keep, keep)], P[keep] / 100)
    return pd.Series(b * (theta[f] - theta[t]) * 100, index=[f"{a}-{c}" for a, c in zip(LINES.bus0, LINES.bus1)])
    ### END SOLUTION

In [3]:
own = dc_flows()
n = croatia_pypsa()
n.lpf()
assert np.abs(own.to_numpy() - n.lines_t.p0.iloc[0].to_numpy()).max() < 1e-6
own.round(1)

Zagreb-Rijeka    -352.7
Rijeka-Velebit   -152.7
Velebit-Split     147.3
Zagreb-Split     -122.1
Zagreb-Osijek      74.9
Osijek-Split     -175.1
dtype: float64

**2. Losses from DC flows**

DC is lossless, but its flows can estimate the losses: on a line, losses ≈ r · P² / V² (with V = 1 p.u.). Write `dc_loss_estimate(flows_mw)` returning the total estimated losses (MW) for the Croatian network, and compare with the AC losses from pandapower.

In [4]:
def dc_loss_estimate(flows_mw: pd.Series) -> float:
    """Sum over lines of r_pu * P_pu^2, in MW (100 MVA base, 400 kV)."""
    ### BEGIN SOLUTION
    r_pu = R_KM * LINES.km.to_numpy() / (400 ** 2 / 100)
    return float((r_pu * (flows_mw.to_numpy() / 100) ** 2).sum() * 100)
    ### END SOLUTION

In [5]:
estimate = dc_loss_estimate(own)
net = croatia_pandapower()
pp.runpp(net, numba=False)
actual = net.res_line.pl_mw.sum()
assert abs(estimate - actual) / actual < 0.15, "within 15 % of the AC losses"
print(f"DC-based estimate {estimate:.2f} MW, AC losses {actual:.2f} MW ({100 * (estimate / actual - 1):+.1f} %)")

DC-based estimate 6.74 MW, AC losses 7.00 MW (-3.7 %)


### Core

**3. DC angles as a starting point**

Newton–Raphson from a flat start begins at all angles zero. Start it instead from the **DC angles** (and the set-point voltages at slack and PV buses, 1 p.u. elsewhere). For the IEEE 14-bus system at load scales [1.0, 2.0, 3.0, 4.0] (generators scaled too), compare the iterations from a flat start and from the DC start. Write `dc_start(load_scale)`; use pandapower's `rundcpp` for the DC angles and map them into the internal order with `order`.

In [6]:
def dc_start(load_scale: float) -> dict:
    """Newton iterations from a flat start and from the DC solution."""
    ### BEGIN SOLUTION
    net, d = case14_data(load_scale)
    pp.rundcpp(net)
    theta = np.zeros(len(d["S"]))
    theta[d["order"][net.bus.index]] = np.radians(net.res_bus.va_degree.to_numpy())
    flat = d["v_set"].astype(complex)
    warm = d["v_set"] * np.exp(1j * theta)
    V1, it_flat = newton_raphson(d["Y"], d["S"], flat, d["pv"], d["pq"])
    V2, it_dc = newton_raphson(d["Y"], d["S"], warm, d["pv"], d["pq"])
    assert np.abs(V1 - V2).max() < 1e-8
    return {"flat start": it_flat, "DC start": it_dc}
    ### END SOLUTION

In [7]:
starts = pd.DataFrame({s: dc_start(s) for s in [1.0, 2.0, 3.0, 4.0]}).T.rename_axis("load scale")
assert (starts["DC start"] <= starts["flat start"]).all()
starts

            flat start  DC start
load scale                      
1.0                  4         4
2.0                  4         4
3.0                  5         4
4.0                  7         6

**4. An overload DC cannot see**

Give the Zagreb–Rijeka line (position 0) a thermal rating of 0.55 kA. Sweep the Zagreb voltage set point over [1.00, 1.01, …, 1.05] and report, for each, the line's active and reactive flow (AC), its loading by DC (|P_DC| / (√3 · 400 kV · 0.55 kA)) and its loading by AC (pandapower's `loading_percent`). From which set point does AC report an overload that DC misses? Write `hidden_overload(setpoints)`.

In [8]:
def hidden_overload(setpoints) -> pd.DataFrame:
    """Zagreb-Rijeka flows and loadings (DC and AC) for each Zagreb voltage set point."""
    ### BEGIN SOLUTION
    rows = {}
    rating_mva = np.sqrt(3) * 400 * 0.55
    for v in setpoints:
        net = croatia_pandapower(v_zagreb=v, rating_ka={0: 0.55})
        pp.runpp(net, numba=False)
        p_ac, q_ac, ac_loading = net.res_line.p_from_mw[0], net.res_line.q_from_mvar[0], net.res_line.loading_percent[0]
        pp.rundcpp(net)
        rows[v] = {"P_AC_MW": p_ac, "Q_AC_Mvar": q_ac, "DC_loading_%": 100 * abs(net.res_line.p_from_mw[0]) / rating_mva,
                   "AC_loading_%": ac_loading}
    return pd.DataFrame(rows).T.rename_axis("Zagreb set point (p.u.)")
    ### END SOLUTION

In [9]:
sweep = hidden_overload(np.round(np.arange(1.00, 1.051, 0.01), 2))
assert sweep["DC_loading_%"].max() < 100 and sweep["AC_loading_%"].max() > 100
missed = sweep.index[(sweep["AC_loading_%"] > 100) & (sweep["DC_loading_%"] < 100)]
print("overloads DC misses at set points:", list(missed))
sweep.round(1)

overloads DC misses at set points: [1.03, 1.04, 1.05]


                         P_AC_MW  Q_AC_Mvar  DC_loading_%  AC_loading_%
Zagreb set point (p.u.)                                                
1.00                      -348.4        9.1          92.6          93.2
1.01                      -348.7       47.4          92.6          95.3
1.02                      -348.8       86.4          92.6          98.3
1.03                      -348.9      126.2          92.6         102.3
1.04                      -349.0      166.8          92.6         107.0
1.05                      -348.9      208.1          92.6         112.5

**5. Interpretation**

How good was the loss estimate, and why does it work although DC is lossless? How much does a DC start save Newton–Raphson, and when? Why can DC not see the overload in task 4, and which kind of study must therefore use AC?

<!-- BEGIN SOLUTION -->
The DC-based estimate comes within 4 % of the AC losses (6.74 against 7.00 MW). It works because on a line with small reactive flow and voltages near 1 p.u. the current is almost proportional to the active flow, so the losses r · I² are almost r · P². DC gets P nearly right, and the losses follow — the model is lossless, but its flows still carry the information to estimate them. That is how some market models add losses back after the fact. The estimate is a little low because it leaves out the reactive part of the current.

A DC start saves little here: the same number of iterations at normal loads, one iteration fewer at three and four times the load. Newton–Raphson from a flat start is already efficient on a well-conditioned network; a better start matters more for difficult cases near the loadability limit, where a flat start can fail altogether — pandapower offers `init="dc"` for that reason.

DC cannot see the overload in task 4 because it has thrown away exactly what causes it: the active flow on Zagreb–Rijeka hardly changes (about 349 MW at every set point), but raising the voltage at Zagreb drives up to 208 Mvar of reactive power along the line, and the current — which is what heats the conductor — grows with both. From a set point of 1.03 p.u. the line is above its rating in AC (102 %) while DC still reports 92.6 %. Any study that must certify thermal limits or voltages — security assessment, connection studies, the real-time operation of a TSO — has to use AC; DC is fit for market clearing and screening, where its errors are acceptable or later corrected.
<!-- END SOLUTION -->

### Challenge

**6. Does DC rank the outages right?**

Security screening asks: which single line outage stresses the network most? For each of the six lines of the Croatian network, take it out of service and compute the largest line flow (MW) with AC and with DC. Rank the outages by each method and compare. Write `n_minus_1()`.

In [10]:
def n_minus_1() -> pd.DataFrame:
    """Largest line flow after each single line outage, AC and DC, with ranks."""
    ### BEGIN SOLUTION
    rows = {}
    for k, line in LINES.iterrows():
        net = croatia_pandapower(out_of_service=k)
        pp.runpp(net, numba=False)
        ac = net.res_line.p_from_mw.abs().max()
        pp.rundcpp(net)
        dc = net.res_line.p_from_mw.abs().max()
        rows[f"{line.bus0}-{line.bus1} out"] = {"AC_max_MW": ac, "DC_max_MW": dc}
    out = pd.DataFrame(rows).T
    out["AC_rank"] = out.AC_max_MW.rank(ascending=False, method="min")
    out["DC_rank"] = out.DC_max_MW.rank(ascending=False, method="min")
    return out
    ### END SOLUTION

In [11]:
screen = n_minus_1()
assert len(screen) == 6
assert (screen.DC_max_MW - screen.AC_max_MW).abs().max() < 20
screen.round(1)

                    AC_max_MW  DC_max_MW  AC_rank  DC_rank
Zagreb-Rijeka out       499.2      500.0      1.0      1.0
Rijeka-Velebit out      300.0      300.0      6.0      6.0
Velebit-Split out       492.6      500.0      2.0      1.0
Zagreb-Split out        417.5      425.0      4.0      4.0
Zagreb-Osijek out       314.9      319.4      5.0      5.0
Osijek-Split out        425.4      430.6      3.0      3.0

**7. Interpretation**

Did DC rank the outages in the same order as AC? What does that suggest for how screening studies combine DC and AC power flows?

<!-- BEGIN SOLUTION -->
Yes — DC ranks the six outages in the same order as AC, except that it ties the two worst at exactly 500 MW (losing Zagreb–Rijeka or Velebit–Split forces the whole export of Rijeka and Velebit onto one path, and in a lossless model that is exactly their 500 MW of net injection). AC separates them, at 499.2 and 492.6 MW, because losses along the path differ. The flows differ by at most 7.4 MW.

That is the basis of the usual screening practice: run thousands of cheap DC (or fast-decoupled) contingency cases to find the few that come close to a limit, then confirm those with full AC power flows, which also check voltages and reactive flows. DC is trusted to point to the dangerous cases, not to certify that they are safe — as task 4 showed, a case DC calls safe can still be overloaded once reactive power is counted. Lesson 6 builds that screening.
<!-- END SOLUTION -->